# 📊 ASER Dataset — EDA & Train/Dev/Test Splits
**Project:** Indian Children Speech Recognition (ICASSP)

**What this notebook does:**
- Loads the manifest built in Notebook 01
- Computes exact audio duration for every clip
- Prints full dataset statistics (language, region, age, reading level)
- Creates speaker-independent train / dev / test splits
- Saves 6 CSV files ready for model training

**Run after:** `01_dataset_preparation.ipynb`


## Cell 1 — Imports and Paths


In [ ]:
import csv
import subprocess
import random
from pathlib import Path
from collections import defaultdict

random.seed(42)   # Fixed seed = reproducible splits every time

MANIFEST = Path("/home/hp/Indain_children_spech/ASER-Dataset/manifest.csv")
OUT_DIR  = Path("/home/hp/Indain_children_spech/ASER-Dataset")

print("Paths configured.")
print(f"Manifest: {MANIFEST}")


## Cell 2 — Load Manifest
Load all 81K rows from the manifest CSV into memory.


In [ ]:
records = []
with open(MANIFEST, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        records.append(row)

print(f"Loaded {len(records):,} records.")
print(f"Columns: {list(records[0].keys())}")


## Cell 3 — Compute Exact Duration with ffprobe
`ffprobe` reads the audio file header and returns the duration in seconds.
We do this for every clip so we can report honest hours, not estimates.

⏱ This takes ~8-10 minutes for 81K files.


In [ ]:
def get_duration(path):
    """Returns duration in seconds for an audio file using ffprobe."""
    try:
        result = subprocess.run(
            ["ffprobe", "-v", "error",
             "-show_entries", "format=duration",
             "-of", "default=noprint_wrappers=1:nokey=1", path],
            capture_output=True, text=True, timeout=10
        )
        val = result.stdout.strip()
        return float(val) if val else 0.0
    except Exception:
        return 0.0

print("Computing durations... (this takes ~8-10 minutes)")
for i, r in enumerate(records):
    r["duration_sec"] = get_duration(r["audio_path"])
    if (i+1) % 10000 == 0:
        print(f"  {i+1:,} / {len(records):,} done...")

# Save durations back to manifest
with open(MANIFEST, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(records[0].keys()))
    writer.writeheader()
    writer.writerows(records)

total_hrs = sum(float(r["duration_sec"]) for r in records) / 3600
print(f"\nTotal dataset duration: {total_hrs:.2f} hours")


## Cell 4 — Overall Statistics


In [ ]:
total_clips    = len(records)
total_secs     = sum(float(r["duration_sec"]) for r in records)
total_hrs      = total_secs / 3600
total_children = len(set(r["child_id"] for r in records))

print("=" * 50)
print("  ASER DATASET — OVERVIEW")
print("=" * 50)
print(f"  Total clips      : {total_clips:,}")
print(f"  Unique children  : {total_children:,}")
print(f"  Total duration   : {total_hrs:.2f} hours")
print(f"  Avg clip length  : {total_secs/total_clips:.2f} seconds")


## Cell 5 — Duration by Language


In [ ]:
lang_dur = defaultdict(float)
lang_cnt = defaultdict(int)
for r in records:
    lang_dur[r["script_language"]] += float(r["duration_sec"])
    lang_cnt[r["script_language"]] += 1

print(f"{'Language':<30} {'Clips':>7}  {'Hours':>8}")
print("-" * 50)
for lang in sorted(lang_dur, key=lambda x: -lang_dur[x]):
    print(f"{lang:<30} {lang_cnt[lang]:>7,}  {lang_dur[lang]/3600:>8.2f}")


## Cell 6 — Duration by Region


In [ ]:
reg_dur = defaultdict(float)
reg_cnt = defaultdict(int)
for r in records:
    reg_dur[r["region"]] += float(r["duration_sec"])
    reg_cnt[r["region"]] += 1

print(f"{'Region':<30} {'Clips':>7}  {'Hours':>8}")
print("-" * 50)
for reg in sorted(reg_dur, key=lambda x: -reg_dur[x]):
    print(f"{reg:<30} {reg_cnt[reg]:>7,}  {reg_dur[reg]/3600:>8.2f}")


## Cell 7 — Duration by Reading Level


In [ ]:
lvl_dur = defaultdict(float)
lvl_cnt = defaultdict(int)
for r in records:
    lvl_dur[r["reading_level"]] += float(r["duration_sec"])
    lvl_cnt[r["reading_level"]] += 1

print(f"{'Reading Level':<35} {'Clips':>7}  {'Hours':>8}  ASR Useful?")
print("-" * 65)
asr_levels = {"Paragraph", "Story", "Sentence (English)"}
for lvl in sorted(lvl_dur, key=lambda x: -lvl_dur[x]):
    useful = "YES ✓" if lvl in asr_levels else "-"
    print(f"{lvl:<35} {lvl_cnt[lvl]:>7,}  {lvl_dur[lvl]/3600:>8.2f}  {useful}")


## Cell 8 — Age Group Distribution


In [ ]:
age_dur = defaultdict(float)
age_cnt = defaultdict(int)
for r in records:
    age_dur[r["age_group"]] += float(r["duration_sec"])
    age_cnt[r["age_group"]] += 1

print(f"{'Age Group':<25} {'Clips':>7}  {'Hours':>8}")
print("-" * 45)
for age in sorted(age_cnt.keys()):
    print(f"{age:<25} {age_cnt[age]:>7,}  {age_dur[age]/3600:>8.2f}")


## Cell 9 — Reading Quality (Correct vs Incorrect)


In [ ]:
correct   = [r for r in records if r["is_correct"] == "True"]
incorrect = [r for r in records if r["is_correct"] == "False"]

print(f"Correct readings   : {len(correct):,}  ({len(correct)/len(records)*100:.1f}%)")
print(f"Incorrect readings : {len(incorrect):,}  ({len(incorrect)/len(records)*100:.1f}%)")
print()
print("Note: Incorrect readings are NOT discarded.")
print("They contain real child mispronunciation patterns — valuable for training robustness.")


## Cell 10 — ASR-Useful Subset Summary
Only Paragraph + Story + Sentence clips are used for ASR model training.
Single letters and words have no linguistic context.


In [ ]:
asr_levels   = {"Paragraph", "Story", "Sentence (English)"}
asr_records  = [r for r in records if r["reading_level"] in asr_levels]
asr_hrs      = sum(float(r["duration_sec"]) for r in asr_records) / 3600
asr_children = len(set(r["child_id"] for r in asr_records))

print(f"ASR-useful clips    : {len(asr_records):,}")
print(f"ASR-useful duration : {asr_hrs:.2f} hours")
print(f"Unique children     : {asr_children:,}")

asr_lang = defaultdict(int)
for r in asr_records:
    asr_lang[r["script_language"]] += 1
print()
for lang, cnt in sorted(asr_lang.items(), key=lambda x: -x[1]):
    print(f"  {lang:<25} {cnt:,} clips")


## Cell 11 — Create Train / Dev / Test Splits

### Why split by child, not by clip?
If we split randomly by clip, the same child's voice appears in BOTH train and test.
The model hears that voice during training → unfair advantage at test time → fake good results.

**Splitting by child_id guarantees the test set contains only voices the model has NEVER heard.**
This is called a speaker-independent evaluation — the standard for publishable ASR research.

### Strategy
- Group children by region (RJ, UP, MH) for balanced regional representation
- Within each region: 80% train / 10% dev / 10% test
- All clips of a child go entirely into ONE split (no leakage)


In [ ]:
region_children = defaultdict(list)
for r in records:
    region_children[r["region"]].append(r["child_id"])

region_unique = {reg: list(set(ids)) for reg, ids in region_children.items()}

train_ids, dev_ids, test_ids = set(), set(), set()

for reg, children in region_unique.items():
    random.shuffle(children)
    n      = len(children)
    n_test = max(1, int(n * 0.10))
    n_dev  = max(1, int(n * 0.10))
    test_ids.update(children[:n_test])
    dev_ids.update(children[n_test : n_test + n_dev])
    train_ids.update(children[n_test + n_dev :])

train_records = [r for r in records if r["child_id"] in train_ids]
dev_records   = [r for r in records if r["child_id"] in dev_ids]
test_records  = [r for r in records if r["child_id"] in test_ids]

print("Split sizes:")
for name, recs in [("TRAIN", train_records), ("DEV", dev_records), ("TEST", test_records)]:
    hrs      = sum(float(r["duration_sec"]) for r in recs) / 3600
    children = len(set(r["child_id"] for r in recs))
    print(f"  {name:<6}: {len(recs):>6,} clips | {children:>4} children | {hrs:.2f} hrs")

# Leakage check
print(f"\nSpeaker leakage check:")
print(f"  Train ∩ Test : {len(train_ids & test_ids)} (must be 0)")
print(f"  Train ∩ Dev  : {len(train_ids & dev_ids)} (must be 0)")
print(f"  Dev   ∩ Test : {len(dev_ids   & test_ids)} (must be 0)")


## Cell 12 — Save All Split CSVs


In [ ]:
fieldnames = list(records[0].keys())

# Full splits (all reading levels)
for name, recs in [("train", train_records), ("dev", dev_records), ("test", test_records)]:
    out = OUT_DIR / f"{name}.csv"
    with open(out, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(recs)
    print(f"Saved {name}.csv  →  {len(recs):,} clips")

print()

# ASR-only splits (connected speech: paragraph + story + sentence)
for name, recs in [("train", train_records), ("dev", dev_records), ("test", test_records)]:
    asr_recs = [r for r in recs if r["reading_level"] in asr_levels]
    hrs      = sum(float(r["duration_sec"]) for r in asr_recs) / 3600
    out      = OUT_DIR / f"asr_{name}.csv"
    with open(out, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(asr_recs)
    print(f"Saved asr_{name}.csv  →  {len(asr_recs):,} clips  |  {hrs:.2f} hrs")

print("\nAll done! Dataset fully prepared for ASR training.")
